# Stage 01 - Data Ingestion: Text Splitting Techniques
### Lecture 11: Character, Recursive, and Token Text Splitters

This notebook implements core document chunking strategies:
1. `CharacterTextSplitter`: Splitting on fixed delimiters.
2. `RecursiveCharacterTextSplitter`: Hierarchical splitting preserving natural paragraphs and sentences.
3. `TokenTextSplitter`: Aligning chunk boundaries with LLM tokenizer constraints.
4. Preserving metadata and tracking chunk indices across document splits.


In [1]:
# Step 1: Environment Setup and Sample Corpus Definition
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)
from langchain_core.documents import Document

sample_text = """Retrieval-Augmented Generation (RAG) is an architectural pattern that optimizes language model output.
By referencing an authoritative knowledge base outside of its training data sources before generating a response, RAG extends the already powerful capabilities of LLMs to specific domains or an organization's internal knowledge base, all without the need to retrain the model.

Traditional search engines rely purely on lexical matching such as BM25 or TF-IDF.
Modern vector databases enable semantic similarity search using dense vector representations.
Combining dense and sparse representations yields hybrid retrieval, which maximizes both keyword precision and semantic recall."""

source_doc = Document(
    page_content=sample_text,
    metadata={"source": "rag_architecture_overview.txt", "doc_type": "technical"}
)

print(f"[INFO] Initialized raw sample document ({len(source_doc.page_content)} characters).")


[INFO] Initialized raw sample document (685 characters).


In [2]:
# Step 2: RecursiveCharacterTextSplitter (Recommended Production Standard)
# Hierarchically attempts splitting on: ["\n\n", "\n", " ", ""]
chunk_size = 200
chunk_overlap = 40

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    is_separator_regex=False
)

recursive_chunks = recursive_splitter.split_documents([source_doc])

print(f"[INFO] Recursive splitter produced {len(recursive_chunks)} chunks (size={chunk_size}, overlap={chunk_overlap}):")
for idx, chunk in enumerate(recursive_chunks, start=1):
    print(f"  Chunk {idx} ({len(chunk.page_content)} chars): {chunk.page_content[:65]}...")
    print(f"    Metadata: {chunk.metadata}")

# Assertions
assert len(recursive_chunks) >= 3, "Expected at least 3 chunks with given size and overlap."
assert all(len(c.page_content) <= chunk_size + 20 for c in recursive_chunks), "Chunk size must respect configured boundaries."
assert all(c.metadata["source"] == source_doc.metadata["source"] for c in recursive_chunks), "Metadata must be preserved across chunks."


[INFO] Recursive splitter produced 5 chunks (size=200, overlap=40):
  Chunk 1 (102 chars): Retrieval-Augmented Generation (RAG) is an architectural pattern ...
    Metadata: {'source': 'rag_architecture_overview.txt', 'doc_type': 'technical'}
  Chunk 2 (193 chars): By referencing an authoritative knowledge base outside of its tra...
    Metadata: {'source': 'rag_architecture_overview.txt', 'doc_type': 'technical'}
  Chunk 3 (116 chars): of LLMs to specific domains or an organization's internal knowled...
    Metadata: {'source': 'rag_architecture_overview.txt', 'doc_type': 'technical'}
  Chunk 4 (176 chars): Traditional search engines rely purely on lexical matching such a...
    Metadata: {'source': 'rag_architecture_overview.txt', 'doc_type': 'technical'}
  Chunk 5 (127 chars): Combining dense and sparse representations yields hybrid retrieva...
    Metadata: {'source': 'rag_architecture_overview.txt', 'doc_type': 'technical'}


In [3]:
# Step 3: TokenTextSplitter (Tokenizer-Aware Splitting)
# Splitting directly based on BPE / tiktoken token limits to avoid truncation in LLM context windows
token_splitter = TokenTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)

token_chunks = token_splitter.split_documents([source_doc])
print(f"[INFO] Token splitter produced {len(token_chunks)} token-aligned chunks:")
for idx, chunk in enumerate(token_chunks, start=1):
    print(f"  Token Chunk {idx}: {chunk.page_content[:60]}...")

# Assertions
assert len(token_chunks) >= 2, "Token splitter must generate valid chunks."
print("[SUCCESS] All text splitting strategies validated with verified metadata propagation.")


[INFO] Token splitter produced 3 token-aligned chunks:
  Token Chunk 1: Retrieval-Augmented Generation (RAG) is an architectural pat...
  Token Chunk 2: AG extends the already powerful capabilities of LLMs to spec...
  Token Chunk 3:  matching such as BM25 or TF-IDF.
Modern vector databases en...
[SUCCESS] All text splitting strategies validated with verified metadata propagation.
